In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("sample_data/dataset_limpo.parquet")
# Supondo que o seu DataFrame consolidado das 85 mil linhas se chame 'df'
# Vamos padronizar os nomes das colunas para minúsculas caso ainda não estejam:
df.columns = [col.lower() for col in df.columns]

# 1. Limpeza básica de espaços em strings e conversão para minúsculas
df['municipio'] = df['municipio'].str.strip().str.lower()
df['uf'] = df['uf'].str.strip().str.lower()
df['especie_arma'] = df['especie_arma'].str.strip().str.lower()

# 2. Criar variáveis indicadoras (One-Hot Encoding) para as principais espécies de armas
# Exemplo: transformar 'espingarda', 'pistola', 'revólver' em colunas binárias/contadoras
especies_frequentes = df['especie_arma'].value_counts().head(5).index.tolist()
df_filtrado = df[df['especie_arma'].isin(especies_frequentes)]

# Tabela pivotada para contar a quantidade de cada tipo de arma por Município e UF
pivot_especies = pd.crosstab(
    [df['municipio'], df['uf']], 
    df['especie_arma'], 
    values=df['total'], 
    aggfunc='sum'
).fillna(0)

# 3. Consolidar o total geral de armas apreendidas e métricas por município
perfil_municipal = df.groupby(['municipio', 'uf']).agg(
    total_apreensoes=('total', 'sum'),
    ocorrencias_distintas=('tipo_ocorrencia', 'count')
).reset_index()

# Juntar as contagens de espécies de armas ao perfil municipal
perfil_municipal = perfil_municipal.merge(pivot_especies, on=['municipio', 'uf'])

print(f"Dimensões da Matriz de Perfis Municipais: {perfil_municipal.shape}")
display(perfil_municipal.head())

KeyError: 'espécie'